In [39]:
import pandas as pd

Purpose

Test if models trained on  real bulk are good at learning the simulated excitatory neuron enriched functions

In [40]:
df = pd.read_csv("/space/grp/aadrian/Pseudobulk_Function_Pipeline_HighRes/bin/bulkEGADPipeline/results/realbulk_simexilabels/EGAD/melted_dfs/Brain_split.csv_melted_EGADs.csv.gz").reset_index()
df['group'] = df.loc[:,'index'].str.split("_").str[2]
df = df.query("group!='Q1'")
df

,level_0,Unnamed: 0,index,bootstrap,auc,group
999,999,999,SIMGO_1_Q2,44,0.489572,Q2
1000,1000,1000,SIMGO_2_Q2,44,0.452248,Q2
1001,1001,1001,SIMGO_3_Q2,44,0.489362,Q2
1002,1002,1002,SIMGO_4_Q2,44,0.527717,Q2
1003,1003,1003,SIMGO_5_Q2,44,0.486215,Q2
...,...,...,...,...,...,...
149845,149845,2992,SIMGO_995_Q3,42,0.807613,Q3
149846,149846,2993,SIMGO_996_Q3,42,0.685613,Q3
149847,149847,2994,SIMGO_997_Q3,42,0.828923,Q3
149848,149848,2995,SIMGO_998_Q3,42,0.734947,Q3


In [67]:
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

df['go_num'] = df['index'].str.split('_').str[1]
df_agg = df.groupby(['go_num', 'group'])['auc'].mean().reset_index()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    lme_model = smf.mixedlm('auc ~ C(group)', df_agg, groups=df_agg['go_num'])
    lme_result = lme_model.fit(method='lbfgs')

failed = any(issubclass(w.category, (ConvergenceWarning, UserWarning)) for w in caught)

if failed:
    print("NOTE: Mixed effects model failed to converge (singular random effects). Falling back to t-test.")
    result = None
else:
    result = lme_result
    print(result.summary())

NOTE: Mixed effects model failed to converge (singular random effects). Falling back to t-test.


In [47]:
df

,level_0,Unnamed: 0,index,bootstrap,auc,group,go_id
999,999,999,SIMGO_1_Q2,44,0.489572,Q2,SIMGO_1_Q2
1000,1000,1000,SIMGO_2_Q2,44,0.452248,Q2,SIMGO_2_Q2
1001,1001,1001,SIMGO_3_Q2,44,0.489362,Q2,SIMGO_3_Q2
1002,1002,1002,SIMGO_4_Q2,44,0.527717,Q2,SIMGO_4_Q2
1003,1003,1003,SIMGO_5_Q2,44,0.486215,Q2,SIMGO_5_Q2
...,...,...,...,...,...,...,...
149845,149845,2992,SIMGO_995_Q3,42,0.807613,Q3,SIMGO_995_Q3
149846,149846,2993,SIMGO_996_Q3,42,0.685613,Q3,SIMGO_996_Q3
149847,149847,2994,SIMGO_997_Q3,42,0.828923,Q3,SIMGO_997_Q3
149848,149848,2995,SIMGO_998_Q3,42,0.734947,Q3,SIMGO_998_Q3


In [ ]:
if result is not None:
    coef_df = pd.DataFrame({
        'coef': result.fe_params,
        'std_err': result.bse_fe,
        'z': result.tvalues,
        'p_value': result.pvalues
    }).dropna()
    coef_df.to_csv("results/learnability_mixedlm_coefficients.csv")
    display(coef_df)
else:
    print("Skipping: mixed model did not converge.")

In [ ]:
from plotnine import *

df_plot = df[df['group'] != 'Q1'].copy()

p = (
    ggplot(
        df_plot,
        aes(x='group', y='auc')
    )
    + geom_boxplot(outlier_alpha=0, fill='steelblue')
    + scale_x_discrete(labels=['Non-enriched', 'Exci-Enriched'])
    + labs(
        x='',
        y='Learnability (AUROC)'
    )
    + theme_classic()
    + theme(
        figure_size=(1.5, 1),
        axis_text_x=element_text(rotation=0, size=7),
        axis_text_y=element_text(rotation=0,size=7),
        axis_title_y=element_text(size=8),
        axis_title_x=element_text(size=8),
        legend_position='none'
    )
    + scale_color_brewer(type='qual', palette='Dark2')
    +coord_flip()
) 

p.save("results/learnability_boxplot.png", dpi=300)
p.save("results/learnability_boxplot.pdf")

p

NameError: name 'merged_bulk' is not defined

In [ ]:
if result is not None:
    intercept = result.fe_params['Intercept']
    slopes_df = pd.DataFrame({
        'group': ['Q2', 'Q3'],
        'predicted_mean_auc': [
            intercept,
            intercept + result.fe_params['C(group)[T.Q3]'],
        ]
    })
    slopes_df.to_csv("results/slopes.csv", index=False)
    display(slopes_df)
else:
    print("Skipping: mixed model did not converge.")

In [ ]:
from scipy import stats

q2 = df_agg[df_agg['group'] == 'Q2']['auc']
q3 = df_agg[df_agg['group'] == 'Q3']['auc']

t_stat, p_val = stats.ttest_ind(q2, q3)

ttest_df = pd.DataFrame({
    'comparison': ['Q2 vs Q3'],
    'mean_Q2': [q2.mean()],
    'mean_Q3': [q3.mean()],
    't_stat': [t_stat],
    'p_value': [p_val]
})

ttest_df.to_csv("results/ttest_results.csv", index=False)
ttest_df